##### **Make sure to be in the conda environment**

In [ ]:
import sys
print(sys.executable)

#### 1. Data Cleaning
##### **Objective:** Load, explore and clean the datasets from Steam games and reviews 
##### to create a clean dataset and begin the exploratory analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

ds = load_dataset("FronkonGames/steam-games-dataset")
games = ds['train'].to_pandas()

# Verificar que positive llegue bien
print(games[['positive', 'negative']].describe())
print("\nTop 5 por positive:")
print(games[['name', 'positive', 'negative']].sort_values('positive', ascending=False).head())
pd.set_option('display.max_columns', None) #To see all the columns for a better EDA

README.md: 0.00B [00:00, ?B/s]

D:\Anaconda\envs\steam_games_analysis\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\death\.cache\huggingface\hub\datasets--FronkonGames--steam-games-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

In [ ]:
games = pd.read_csv('../data/raw/games.csv', 
                    encoding='utf-8', # in case theres special characters 
                    quotechar='"', # in case there's ',' in between '"' treat it like one cell
                    on_bad_lines='skip', # ignore instead of breaking 
                    index_col=False) # to prevent AppID being used as the default index

reviews = pd.read_csv('../data/raw/dataset.csv',
                      encoding='utf-8',
                      quotechar='"',
                      on_bad_lines='skip',
                      nrows=500_000, #Limit the rows so it doesnt take forever to read
                      index_col=False) 

print(f'Games: {games.shape}')
print(f'Reviews: {reviews.shape}')
# DIAGNOSTICO — correr en 01_data_cleaning.ipynb

# 1. Ver el raw ANTES de cualquier modificacion
print("RAW original:")
print(games[['Positive', 'Negative']].describe())

#### **Explore both datasets**

##### **Explore the games dataset**

In [ ]:
games.head(5)

In [ ]:
games.info()

In [ ]:
games.describe()

In [ ]:
print('Nulls per column in games:')
print(games.isnull().sum())

##### **Explore the reviews dataset**

In [ ]:
reviews.head()

In [ ]:
reviews.info()

In [ ]:
reviews.describe()

In [ ]:
print('Nulls per column in reviews:')
print(reviews.isnull().sum())

In [ ]:
reviews['app_id'].nunique()

#### **Time to clean the datasets**
- **Remove nulls from critical columns**
- **Change data types if necessary**
- **Rename columns if it has ambigous name**

#### **Starting with the games dataset**

In [ ]:
games_clean = games.copy() 
# 2. Ver games_clean justo despues del copy
games_clean = games.copy()
print("\nDespues del copy:")
print(games_clean[['Positive', 'Negative']].describe())
drop_cols = [
    # > 95% nulls
    'Movies', 'Score rank', 'Reviews', 'Notes','Achievements','User score',
    # URLs don't really care
    'Header image', 'Screenshots', 'Metacritic url',
    # Contact Info
    'Website', 'Support url', 'Support email',
    # Too much text, not planning to use it 
    'About the game', 'Full audio languages',
]
games_clean.drop(columns=drop_cols, inplace=True)
print(f"Clean columns: {games_clean.shape[1]}")

In [ ]:
games_clean.dropna(subset =['AppID'], inplace=True)
games_clean['AppID'].isnull().sum()
duplicate = len(games_clean)

games_clean.drop_duplicates(subset = ['AppID'], keep = 'first', inplace = True)
print(f"Duplicates removed: {duplicate - len(games_clean)}")

games_clean.dropna(subset=['Name'], inplace=True)
games_clean['Name'].isnull().sum()

In [ ]:
games_clean['Release date'] = pd.to_datetime(
    games_clean['Release date'], format = '%b %d, %Y', errors = 'coerce')

print(f"Dates not parsed: {games_clean['Release date'].isna().sum()}")
games_clean['Release_year'] = games_clean['Release date'].dt.year
games_clean['Release_month'] = games_clean['Release date'].dt.month 
games_clean['Month_name'] = games_clean['Release date'].dt.month_name() # for dashboards

In [ ]:
owners = games_clean['Estimated owners'].str.replace(',', '', regex = False) #get both nums by replacing
split = owners.str.split(' - ', expand = True) # splitting them by '-', expand to separate in columns 

games_clean['Owners_mid'] = (split[0].astype(int) + split[1].astype(int)) // 2 # cast both as int and get the mean
games_clean.head(5)

In [ ]:
games_clean.drop(columns = ['Estimated owners'] , inplace = True) # drop old column, not needed anymore
games_clean.head(5)

In [ ]:
text_cols = ['Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Supported languages']

for col in text_cols :
    flag_name = f"has_{col.lower().replace(' ','_')}"
    games_clean[flag_name] = games_clean[col].notna().astype(int)
    games_clean[col] = games_clean[col].fillna('')
games_clean.head(5)

In [ ]:
games_clean[['Windows','Mac','Linux']] = games_clean[['Windows','Mac','Linux']].fillna(False) # fill NaN with false, we assume that its not supported

In [ ]:
playtime_cols = ['Average playtime forever', 'Average playtime two weeks',
                 'Median playtime forever', 'Median playtime two weeks']
games_clean[playtime_cols] = games_clean[playtime_cols].fillna(0) # fill NaN with 0, since there's not enough data of playtime 

In [ ]:
games_clean.columns = ( # renaming all columns to have a standarized version
    games_clean.columns 
    .str.strip()
    .str.lower()
    .str.replace(' ','_')
)

print(f"\nFinal shape: {games_clean.shape}")
print(f"Nulls left:\n{games_clean.isnull().sum()[games_clean.isnull().sum() > 0]}")
games_clean.head(3)

In [ ]:
#games_clean.to_csv('../data/processed/games_clean.csv', index =False) # save the new and clean dataset to use later in the EDA
#print('Saved') # print to make sure it's with no errors

In [ ]:
print(games[['Positive','Negative']].describe() )
print(games_clean[['positive','negative']].describe() )